# Phase 0 — Validation PPG (T0-2)

Objectif : charger un **CSV** (format de référence) ou une **vidéo**, obtenir pour chaque image `timestamp_ms, mean_r, mean_g, mean_b`, puis **afficher la courbe du canal rouge**.

Sans données terrain, on utilise `tests/fixtures/sample_ppg_signal.csv` (~30 s, ~72 bpm).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ppg.io import load_rgb_signal, save_red_channel_svg

print("Racine du projet :", ROOT)

## 1. Charger le signal

Par défaut : fixture CSV. Pour une vidéo de collecte, pointe `SIGNAL_PATH` vers `data/raw/*.mp4` (nécessite OpenCV).

In [ ]:
SIGNAL_PATH = ROOT / "tests" / "fixtures" / "sample_ppg_signal.csv"
# SIGNAL_PATH = ROOT / "data" / "raw" / "exemple.mp4"

signal = load_rgb_signal(SIGNAL_PATH)
print(f"Échantillons : {len(signal)}")
print(f"Durée        : {signal.duration_s:.2f} s")
print(signal.head())

## 2. Courbe du canal rouge

Le canal rouge est le choix par défaut du cahier des charges.

In [ ]:
out_svg = ROOT / "data" / "processed" / "t0_2_red_channel.svg"
save_red_channel_svg(signal, out_svg)
print("Courbe enregistrée :", out_svg)

# Affichage interactif si matplotlib est installé
try:
    import matplotlib.pyplot as plt

    t_s = signal.timestamp_ms / 1000.0
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t_s, signal.mean_r, color="#C62828", linewidth=1, label="mean_r")
    ax.set_xlabel("Temps (s)")
    ax.set_ylabel("Moyenne du canal rouge")
    ax.set_title(f"Canal rouge — {SIGNAL_PATH.name}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib non installé : ouvre le SVG ci-dessus dans un navigateur.")

## Critère T0-2

- Charger CSV/vidéo → `timestamp_ms, mean_r, mean_g, mean_b`
- Courbe du canal rouge visible (SVG et/ou matplotlib)

Prochaine tâche (**T0-3**) : rééchantillonnage 30 Hz, detrend, filtre passe-bande.

Vérification rapide sans Jupyter :
```bash
source .venv/bin/activate
python scripts/verify_t0_2.py
```